In [5]:
from bs4 import BeautifulSoup
import requests
import time
import datetime

import smtplib

In [27]:
URL = 'https://www.amazon.com/Funny-Data-Systems-Business-Analyst/dp/B07FNW9FGJ/'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "DNT": "1", 
    "Connection": "close", 
    "Upgrade-Insecure-Requests": "1"
}

# Send request
page = requests.get(URL, headers=headers)

# Parse HTML
soup = BeautifulSoup(page.content, "html.parser")

# Extract Title
title = soup.find("span", {"id": "productTitle"})
title_text = title.get_text(strip=True) if title else "Title not found"

# Extract Price (checking multiple locations)
price = (
    soup.find("span", {"id": "priceblock_ourprice"}) or 
    soup.find("span", {"id": "priceblock_dealprice"}) or 
    soup.find("span", {"class": "a-price-whole"})  # Some prices use this class
)

price_text = price.get_text(strip=True) if price else "Price not found"

print(f"Product: {title_text}")
print(f"Price: {price_text}")


Product: Got Data Funny Business Data Analyst T-Shirt
Price: 12.


In [25]:
price_whole = soup.find("span", {"class": "a-price-whole"})
price_fraction = soup.find("span", {"class": "a-price-fraction"})

if price_whole:
    price_text = price_whole.get_text(strip=True)
    if price_fraction:
        price_text += "." + price_fraction.get_text(strip=True)  # Add cents if available
else:
    price_text = "Price not found"

# Clean up the data
price_text = price_text.strip() if price_text != "Price not found" else price_text
title_text = title_text.strip()

print(f"Product: {title_text}")
print(f"Price: ${price_text}")

Product: Got Data Funny Business Data Analyst T-Shirt
Price: $12..97


In [11]:
# Create a Timestamp for your output to track when data was collected

import datetime

today = datetime.date.today()

print(today)

2025-03-02


In [49]:
# Create CSV and write headers and data into the file

import csv 

header = ['Title', 'Price', 'Date']
data = [title, price, today]


with open('AmazonWebScraperDataset.csv', 'w', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerow(data)

In [51]:
#Now we are appending data to the csv

with open('AmazonWebScraperDataset.csv', 'a+', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow(data)

In [53]:
from datetime import datetime 
import requests
from bs4 import BeautifulSoup
import pandas as pd

def check_price():
    URL = 'https://www.amazon.com/Funny-Data-Systems-Business-Analyst/dp/B07FNW9FGJ'
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1",
        "Connection": "keep-alive"
    }

    page = requests.get(URL, headers=headers)
    soup = BeautifulSoup(page.content, "html.parser")

    # Extract title
    title = soup.select_one("span#productTitle")
    title = title.get_text(strip=True) if title else "Title Not Found"

    # Extract price
    price = soup.select_one("span.a-price-whole")
    price = price.get_text(strip=True) if price else "Price Not Found"

    # Get current date
    date = datetime.today().strftime('%Y-%m-%d')

    # Store in DataFrame
    df = pd.DataFrame([[title, price, date]], columns=['Title', 'Price', 'Date'])
    print(df)

check_price() 


                                          Title Price        Date
0  Got Data Funny Business Data Analyst T-Shirt   12.  2025-03-02


In [ ]:
# Runs check_price after a set time and inputs data into your CSV

while(True):
    check_price()
    time.sleep(86400)

                                          Title Price        Date
0  Got Data Funny Business Data Analyst T-Shirt   12.  2025-03-02


In [59]:
import pandas as pd
df = pd.read_csv(r'C:\Users\ahmed\py_data\Amazon web scraper project\AmazonWebScraperDataset.csv')

print(df)


                                               Title  \
0  <span class="a-size-large product-title-word-b...   
1  <span class="a-size-large product-title-word-b...   
2       Got Data Funny Business Data Analyst T-Shirt   

                                               Price        Date  
0  <span class="a-price-whole">12<span class="a-p...  2025-03-02  
1  <span class="a-price-whole">12<span class="a-p...  2025-03-02  
2                                               12..  2025-03-02  


In [47]:
whole_price = soup.find("span", {"class": "a-price-whole"})
decimal_price = soup.find("span", {"class": "a-price-decimal"})  # Usually just "."

if whole_price:
    whole_price = whole_price.get_text().strip()
else:
    whole_price = "Price not found"

if decimal_price:
    decimal_price = decimal_price.get_text().strip()
else:
    decimal_price = ""

final_price = whole_price + decimal_price  # Combine both parts

print("Price:", final_price)


Price: 12..


In [57]:


# Extract Title and Price
title = soup.find(id='productTitle').get_text().strip() if soup.find(id='productTitle') else "Title Not Found"

# Extract Whole and Decimal Parts
whole_price = soup.select_one("span.a-price-whole")
decimal_price = soup.select_one("span.a-price-decimal")

whole_price = whole_price.get_text().strip() if whole_price else "Price Not Found"
decimal_price = decimal_price.get_text().strip() if decimal_price else ""

final_price = whole_price + decimal_price  # Combine both parts

# Get Current Date
date = datetime.today().strftime('%Y-%m-%d')

# Store in DataFrame
df = pd.DataFrame([[title, final_price, date]], columns=['Title', 'Price', 'Date'])

# Append to CSV
csv_file = 'AmazonWebScraperDataset.csv'

# Open file in append mode
with open(csv_file, 'a+', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow([title, final_price, date])

print("Data saved successfully!")


Data saved successfully!


In [61]:
import os

csv_file = 'AmazonWebScraperDataset.csv'
print(f"Saving data to: {os.path.abspath(csv_file)}")

with open(csv_file, 'a+', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow([title, final_price, date])

print("Data saved successfully! Check the file above.")


Saving data to: C:\Users\ahmed\py_data\Amazon web scraper project\AmazonWebScraperDataset.csv
Data saved successfully! Check the file above.
